# Kiki SLM — Loopper Organic Data Fine-tuning

Upload to `My Drive/kiki-slm/data/sft-data/`:
- `train_trimmed.jsonl`
- `eval_trimmed.jsonl`

Upload to `My Drive/kiki-slm/data/`:
- `gold_100.jsonl`

In [ ]:
%%capture
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os; os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
!uv pip install --system unsloth wandb datasets
!uv pip install --system transformers==5.2.0

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

REPO_DIR = "/content/drive/MyDrive/kiki-slm/repo"
BRANCH = "feat/loopper-organic-data-pipeline"

if os.path.exists(f"{REPO_DIR}/.git"):
    # Force clean + pull — nuke any local changes
    subprocess.run(f"cd {REPO_DIR} && git fetch origin && git checkout -f {BRANCH} && git reset --hard origin/{BRANCH} && git clean -fd", shell=True)
    print(f"Updated: {REPO_DIR} ({BRANCH})")
else:
    subprocess.run(f"git clone -b {BRANCH} https://github.com/BlueSpringsAI/kiki-slm.git {REPO_DIR}", shell=True)
    print(f"Cloned: {REPO_DIR} ({BRANCH})")

%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# ============================================================
# ALL PATHS — edit here, nowhere else
# ============================================================
TRAIN_FILE = "/content/drive/MyDrive/kiki-slm/data/sft-data/train_trimmed.jsonl"
EVAL_FILE  = "/content/drive/MyDrive/kiki-slm/data/sft-data/eval_trimmed.jsonl"
GOLD_FILE  = "/content/drive/MyDrive/kiki-slm/data/gold_100.jsonl"
DRIVE_OUT  = "/content/drive/MyDrive/kiki-slm/adapters/kiki-sft-v1"

import os
for label, path in [("Train", TRAIN_FILE), ("Eval", EVAL_FILE)]:
    assert os.path.exists(path), f"{label} NOT FOUND: {path}"
    print(f"{label}: {os.path.getsize(path)/1024/1024:.1f} MB")
if os.path.exists(GOLD_FILE):
    print(f"Gold:  {sum(1 for l in open(GOLD_FILE) if l.strip())} tickets")
else:
    print(f"Gold:  not found")
print(f"Output: {DRIVE_OUT}")

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files=TRAIN_FILE, split="train")
print(f"Train: {len(ds):,} examples, columns={ds.column_names}")
del ds

## Train

In [ ]:
!python -u scripts/colab_train.py \
    --train-file {TRAIN_FILE} \
    --eval-file {EVAL_FILE} \
    --output-dir {DRIVE_OUT}

## Evaluate

In [ ]:
import glob
checkpoints = sorted(glob.glob(f"{DRIVE_OUT}/checkpoint-*"))
ADAPTER = checkpoints[-1] if checkpoints else DRIVE_OUT
print(f"Using: {ADAPTER}")

!python -u scripts/colab_eval.py \
    --adapter-path {ADAPTER} \
    --gold-file {GOLD_FILE} \
    --output-file {DRIVE_OUT}/eval_results.json